# Stage1 原始意图受限现金回收：训练来源支持

冻结完整Stage1 f696+b472 为父方案，最终 SOXX 硬门。利用已保存的2018–2022原架构0bp Tiered训练参考，只检验上游 raw-risk 意图下可合法增加0.10 SOXX、减少现金的来源支持；不读未来收益、拟合或运行新账户。

In [1]:
import hashlib,json,os,pathlib,shlex,sys
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_raw_intent_cash_recycle_support_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
src=plan['sources'];source=root/src['training_card'];summary=json.loads((source/'summary.json').read_text())
assert sha(source/'summary.json')==src['training_summary_sha256'] and sha(root/'research/stage1_original_0bp_projection_dollar_slack_20261023/summary.json')==src['held_projection_summary_sha256']
assert sha(root/'src/workflow/intraday_recovery_direct_allocation.py')==src['projector_module_sha256']
assert summary['recorder_id']==src['training_recorder_id'] and summary['experiment_id']==src['training_experiment_id'] and summary['native_audit']['status']=='passed' and summary['native_audit']['slippage_bps']==0
assert summary['nas_readback']['verified'] and summary['nas_readback']['sha256']==summary['nas_evidence']['sha256']
tracking=source/'qlib_recorder';provider=(root/src['provider_uri']).resolve();os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(tracking),'default_exp_name':source.name}})
rec=R.get_recorder(experiment_name=source.name,recorder_id=src['training_recorder_id']);assert rec.status=='FINISHED';frame=rec.load_object('inputs/source_frame').reset_index(drop=True);bundle=rec.load_object('paths/reference_0');path=bundle['replay'].reset_index(drop=True);targets=bundle['targets'].reset_index(drop=True)
assert len(frame)==len(path)==len(targets)==1061 and frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(targets.trade_date)
cal=[str(x)[:10] for x in D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')];assert cal==[str(x)[:10] for x in frame.trade_date]
state={'status':'source_support_running','plan_sha256':sha(card/'plan.json'),'source_recorder_object_loads':2,'Qlib_D_calendar_calls':1,'future_return_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
rows=[];maxerr=0.
for i in range(len(frame)):
 f=frame.iloc[i];p=path.iloc[i];saved=targets.iloc[i][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float);sx=float(f.soxl_weight);xx=float(f.soxx_weight);rawcash=float(f.cash_weight);cap=float(p.current_beta_cap)
 assert np.all(np.isfinite([sx,xx,rawcash,cap])) and min(sx,xx,rawcash)>=-1e-9 and abs(sx+xx+rawcash-1)<1e-7
 control=project_legal_weights(sx,xx,cap,minimum_cash_weight=.15,maximum_soxl_weight=.30);err=float(np.max(np.abs(control-saved)));maxerr=max(maxerr,err);assert err<1e-9,(i,err)
 rawbeta=3*sx+xx;eligible=bool(p.execute_open_target and rawcash>=.25-1e-10 and rawbeta+.10<=cap+1e-10 and sx+xx<=.75+1e-10)
 if eligible:
  candidate=project_legal_weights(sx,xx+.10,cap,minimum_cash_weight=.15,maximum_soxl_weight=.30);assert np.max(np.abs(candidate-(saved+np.array([0.,.10,-.10]))))<1e-9,(i,candidate,saved)
 else:candidate=saved
 rows.append({'trade_date':str(f.trade_date)[:10],'year':int(pd.Timestamp(f.trade_date).year),'control_EXECUTE':bool(p.execute_open_target),'raw_soxl_weight':sx,'raw_soxx_weight':xx,'raw_cash_weight':rawcash,'own_state_beta_cap':cap,'raw_beta':rawbeta,'candidate_source_eligible':eligible,'candidate_projected_soxx_weight':float(candidate[1]),'candidate_projected_cash_weight':float(candidate[2])})
table=pd.DataFrame(rows);annual=[{'year':int(y),'calendar_rows':int(len(g)),'control_EXECUTE':int(g.control_EXECUTE.sum()),'candidate_source_eligible':int(g.candidate_source_eligible.sum())} for y,g in table.groupby('year')];n=int(table.candidate_source_eligible.sum());years20=sum(x['candidate_source_eligible']>=20 for x in annual);passed=n>=100 and years20>=3
result={'decision':'source_support_pass_next_physical_local_value_design' if passed else 'source_support_fail_close_fixed_upstream_cash_recycle_preflight','plan_sha256':sha(card/'plan.json'),'source_recorder_id':rec.id,'sessions':len(table),'max_control_projection_parity_error':maxerr,'candidate_add_SOXX_weight':0.10,'candidate_source_eligible':n,'years_with_at_least20':years20,'support_gate_pass':passed,'annual':annual,'new_market_requests':0,'future_return_reads':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'limitation':'Source-level and projection geometry only; no whole-share Tiered settlement, local profit label, receiver fit, 2023+ own-state account, risk feedback or SOXX gate.'};(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
with R.uri_context(str(card/'qlib_recorder')):
 with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
  out=R.get_recorder();out.save_objects(artifact_path='inputs',plan=plan,source_recorder_id=rec.id,source_NAS_sha256=summary['nas_evidence']['sha256']);out.save_objects(artifact_path='analysis',source_support=table,result=result);out.log_metrics(sessions=len(table),source_eligible=n,support_gate_pass=int(passed),future_return_reads=0,model_fits=0,native_accounts=0);rid,eid=out.id,str(out.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary_out={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_id':rec.id,'source_NAS_sha256':summary['nas_evidence']['sha256'],'source_NAS_readback_verified':True,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'remote_hash_verified':True},'candidate_source_eligible':n,'support_gate_pass':passed,'future_return_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/source_support','analysis/result']},'limitations':['Source-level projection support is not funded whole-share execution or local value.','2018-2022 OOF training reference is not full 2023+ Stage1 candidate account.','No SOXX strategy success or independent forward evidence.']};(card/'summary.json').write_text(json.dumps(summary_out,indent=2)+'\n');print(json.dumps({'decision':result['decision'],'eligible':n,'years20':years20,'annual':annual,'recorder_id':rid},ensure_ascii=False))


2026/10/23 14:00:59 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[3218:MainThread](2026-10-23 14:01:00,506) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[3218:MainThread](2026-10-23 14:01:00,508) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[3218:MainThread](2026-10-23 14:01:00,508) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[3218:MainThread](2026-10-23 14:01:00,839) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_raw_intent_cash_recycle_support_20261023.


[3218:MainThread](2026-10-23 14:01:00,841) INFO - qlib.workflow - [exp.py:258] - Experiment 237703368401205795 starts running ...


[3218:MainThread](2026-10-23 14:01:00,913) INFO - qlib.workflow - [recorder.py:345] - Recorder 68c471a3fffd4948abac133875ca4310 starts running under Experiment 237703368401205795 ...


[3218:MainThread](2026-10-23 14:01:01,520) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "source_support_fail_close_fixed_upstream_cash_recycle_preflight", "eligible": 0, "years20": 0, "annual": [{"year": 2018, "calendar_rows": 53, "control_EXECUTE": 53, "candidate_source_eligible": 0}, {"year": 2019, "calendar_rows": 252, "control_EXECUTE": 252, "candidate_source_eligible": 0}, {"year": 2020, "calendar_rows": 253, "control_EXECUTE": 253, "candidate_source_eligible": 0}, {"year": 2021, "calendar_rows": 252, "control_EXECUTE": 252, "candidate_source_eligible": 0}, {"year": 2022, "calendar_rows": 251, "control_EXECUTE": 251, "candidate_source_eligible": 0}], "recorder_id": "68c471a3fffd4948abac133875ca4310"}


# 结论：原始意图受限现金回收的训练来源支持

冻结完整 Stage1 f696+b472 为经济父方案，SOXX 是最终硬门。本卡只读取保存的 2018–2022 原架构 0bp、Tiered 训练参考；原始投影与保存目标 **1061/1061 完全一致**。不读取未来收益、不拟合、不运行新账户。

预注册的动作是在原 EXECUTE 目标上增加 **10% SOXX**、等额减少现金，保持原有 beta 上限、15% 现金下限和 30% SOXL 上限。来源现金至少25%的日子有 **407** 个，但原始名义 beta 在 **541** 日高于自身上限、**520** 日恰等于上限，**0** 日低于上限。因此该训练参考内符合冻结动作的日期是 **0/1061**，支持门失败；原生 Qlib Recorder `68c471a3fffd4948abac133875ca4310` 与 NAS 选定工件冷读回一致。

**决定：关闭这条固定的同上限 +10% SOXX 训练来源版本。** 这揭示的是训练参考的 beta 空间缺失，不是所有现金回收政策的经济无效证明。2023–2026 完整 Stage1 路径有536个 beta 非绑定、原始风险意图受限的 EXECUTE 日，二者来源状态不能直接拼接训练。若改为释放 beta 或以 SOXX 替换部分 SOXL，属于另一政策；已关闭的直接风险层绕过与构成调整版本不能在复用历史上调参重开。尚无合法局部收益标签、信息门、完整自有状态0bp账户、2/5bp或独立前向证据。
